# QUIBC — XAI Analysis

Grad-CAM, Integrated Gradients, LRP attribution, and latent space visualisations.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from quibc.deployment import load_quibc
from quibc.inference import load_image
from quibc import xai as Q
WEIGHTS = '../checkpoints/best_model.weights.h5'
IMG_SIZE, LATENT_CH = 256, 96

## 1. Load Model

In [ ]:
model = load_quibc(WEIGHTS, img_size=IMG_SIZE, latent_ch=LATENT_CH)
print('Model loaded.')

## 2. Load a Test Image

In [ ]:
# Replace with your own image path
IMG_PATH = '../test_images/sample.jpg'
image = load_image(IMG_PATH, IMG_SIZE)
bits, probs = model.encoder(image, training=False)
recon = model.decoder(bits, training=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image.numpy()[0]); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(recon.numpy()[0]); axes[1].set_title('Reconstructed'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 3. Grad-CAM (Decoder Attention)

In [ ]:
heatmap = Q.grad_cam(model, image, layer_name='dec_trans2')
overlay = Q.overlay_grad_cam(image, heatmap)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image.numpy()[0]); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(heatmap, cmap='jet'); axes[1].set_title('Grad-CAM Heatmap'); axes[1].axis('off')
axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
plt.suptitle('Grad-CAM: Decoder Attention on Perceptually Significant Regions', fontsize=12)
plt.tight_layout(); plt.show()

## 4. Integrated Gradients

In [ ]:
ig = Q.integrated_gradients(model, image, steps=50)
Q.plot_integrated_gradients(image, ig)

## 5. LRP Attribution

In [ ]:
relevance = Q.lrp_attribution(model, image)
Q.plot_lrp(image, relevance, recon=recon)

## 6. Encoder Feature Evolution

In [ ]:
Q.visualise_encoder_features(model, image, n_filters=8)

## 7. Latent Space (PCA & t-SNE)

In [ ]:
from quibc.xai import extract_latent_codes, plot_latent_pca, plot_latent_tsne
from quibc.train import build_clic_datasets
_, val_ds = build_clic_datasets(img_size=IMG_SIZE, batch_size=16, cache=False)
codes, errors = extract_latent_codes(model, val_ds, max_batches=10)
print(f'Extracted {len(codes)} latent codes of dim {codes.shape[1]}')

In [ ]:
plot_latent_pca(codes, errors)

In [ ]:
plot_latent_tsne(codes, errors, perplexity=30)